# Servidor Único de Inferencia en Google Colab (M1 + M2 + M3 + Pasajeros)
Ejecuta todas las celdas en orden (`Entorno de ejecución -> Ejecutar todas`). La URL de Cloudflare se registra **automáticamente** en Render.


In [ ]:
!nvidia-smi -L || echo 'Sin GPU: cambia a GPU T4 en Entorno de ejecucion'


In [ ]:
import os
if not os.path.exists('/content/Control_Peaje'):
    !git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje
else:
    !git -C /content/Control_Peaje fetch --all && git -C /content/Control_Peaje reset --hard origin/main
%cd /content/Control_Peaje
!pip -q install -r inferencia/requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/peaje/evidencias /content/drive/MyDrive/peaje/modelos
!python scripts/diagnostico.py > /content/drive/MyDrive/peaje/evidencias/diagnostico_inferencia.txt
!cat /content/drive/MyDrive/peaje/evidencias/diagnostico_inferencia.txt


In [ ]:
import os
from google.colab import userdata
try:
    os.environ['API_KEY'] = (userdata.get('INFERENCE_API_KEY') or '').strip()
except Exception:
    os.environ['API_KEY'] = ''
    print('Aviso: INFERENCE_API_KEY no definido en Secrets; el endpoint aceptara peticiones sin clave.')

os.environ['M2_PESOS'] = '/content/drive/MyDrive/peaje/modelos/m2_seg.pt'
os.environ['M3_PESOS'] = '/content/drive/MyDrive/peaje/modelos/m3_pose.pt'
os.environ['PASAJEROS_PESOS'] = '/content/drive/MyDrive/peaje/modelos/pasajeros_seg.pt'
os.environ['RENDER_WEB_URL'] = 'https://peaje-web.onrender.com'
print('Pesos configurados correctamente.')


In [ ]:
import threading, time, uvicorn
from inferencia.servidor import app

PUERTO = 8000
config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='warning')
servidor = uvicorn.Server(config)
hilo = threading.Thread(target=servidor.run, daemon=True)
hilo.start()
time.sleep(8)
!curl -s http://127.0.0.1:8000/health | head -c 500


In [ ]:
import os, re, subprocess, time, requests
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
log_cf = open('/tmp/cloudflared.log', 'w')
proc_cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=log_cf, stderr=subprocess.STDOUT)
url_tunel = None
for _ in range(35):
    time.sleep(1)
    txt = open('/tmp/cloudflared.log').read()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url_tunel = m.group(0)
        break
print('🔗 URL del túnel Cloudflare:', url_tunel)

if url_tunel:
    time.sleep(3)
    render_url = os.environ.get('RENDER_WEB_URL', 'https://peaje-web.onrender.com').rstrip('/')
    try:
        r_reg = requests.post(
            f'{render_url}/api/v1/inferencia/registrar-url',
            json={'inference_url': url_tunel, 'api_key': os.environ.get('API_KEY', '')},
            timeout=25,
        )
        if r_reg.status_code == 200:
            print('✅ ¡URL conectada AUTOMÁTICAMENTE en Render!:', r_reg.json())
        else:
            print('⚠️ Pega manualmente la URL en', render_url + '/estado -> Respuesta:', r_reg.text)
    except Exception as e:
        print('⚠️ No se pudo auto-registrar, pégala manualmente en /estado:', e)


In [ ]:
import numpy as np, cv2, requests
assert url_tunel, 'No se obtuvo URL de Cloudflare'
print('GET /health:', requests.get(url_tunel + '/health', timeout=10).json()['status'])
img_dummy = np.zeros((320, 480, 3), dtype=np.uint8)
ok, buf = cv2.imencode('.jpg', img_dummy)
headers = {'X-API-Key': os.environ['API_KEY']} if os.environ.get('API_KEY') else {}
r = requests.post(url_tunel + '/infer', files={'imagen': ('test.jpg', buf.tobytes(), 'image/jpeg')}, data={'modo': 'completo'}, headers=headers, timeout=30)
print('POST /infer status:', r.status_code, 'latencia_ms:', r.json().get('latencia_ms'))


In [ ]:
# Celda keep-alive: mantenla ejecutando durante la grabacion o demo
import time, requests
while True:
    try:
        requests.get('http://127.0.0.1:8000/health', timeout=5)
    except Exception:
        pass
    time.sleep(45)
